# COMP4318/5318 Assignment 2: Image Classification

### Group number: ...  , SID1: ... , SID2: ..., SID3: ..., SID4: ...

**Working notebook: additional, non-assessed reference baselines.** This follows the supplied Canvas template headings so it can be integrated later. These baselines do not replace the required first-six-weeks algorithm, MLP, or CNN. Dataset: the **Canvas PathMNIST subset**, not the A1 rice images.

## Setup and dependencies
Dependencies: Python 3.11; NumPy, pandas, Matplotlib, scikit-learn and Jupyter/IPython. See `requirements.yaml`. The common seed is exactly **5318**.

In [ ]:
from pathlib import Path
import platform
import time
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn import __version__ as sklearn_version
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.neighbors import NearestCentroid
from sklearn.metrics import (accuracy_score, balanced_accuracy_score,
                             f1_score, classification_report, confusion_matrix,
                             ConfusionMatrixDisplay)

SEED = 5318
np.random.seed(SEED)
CLASS_NAMES = {
    0: "Adipose", 1: "Background", 2: "Debris",
    3: "Lymphocytes", 4: "Mucus", 5: "Smooth muscle",
    6: "Normal colon mucosa", 7: "Cancer-associated stroma",
    8: "Colorectal adenocarcinoma epithelium"
}
DATA_RELATIVE = Path("Canvas Sources") / "Assignment2Data-1" / "Assignment2Data"
search_roots = [Path.cwd(), *Path.cwd().parents]
candidate_paths = [
    root / relative
    for root in search_roots
    for relative in (DATA_RELATIVE, Path("Assignment2Data"))
]
DATA_DIR = next((p for p in candidate_paths if (p / "X_train.npy").is_file()), None)
if DATA_DIR is None:
    raise FileNotFoundError(
        "Cannot locate Canvas data. Start Jupyter in the repository root "
        "or place the original four .npy files in an Assignment2Data directory."
    )
print("Seed:", SEED, "| Python:", sys.version.split()[0],
      "| NumPy:", np.__version__, "| scikit-learn:", sklearn_version)
print("Machine:", platform.platform())
print("Original Canvas data:", DATA_DIR.resolve())

## 1. Data loading, exploration, and preprocessing
The Canvas files supply original training/test partitions. We create a stratified 80/20 development split **only from original training data**. No test examples or test scores influence feature design, tuning, or choices.

In [ ]:
# These are the original Canvas training/test partitions, not the public MedMNIST split.
X_train = np.load(DATA_DIR / "X_train.npy", mmap_mode="r", allow_pickle=False)
y_train = np.load(DATA_DIR / "y_train.npy", allow_pickle=False).reshape(-1)
X_test = np.load(DATA_DIR / "X_test.npy", mmap_mode="r", allow_pickle=False)
y_test = np.load(DATA_DIR / "y_test.npy", allow_pickle=False).reshape(-1)

assert X_train.ndim == 4 and X_test.ndim == 4, "Expected NHWC image arrays."
assert X_train.shape[1:] == X_test.shape[1:], "Train/test image dimensions differ."
assert X_train.shape[-1] == 3, "The PathMNIST input should contain three RGB channels."
assert len(X_train) == len(y_train) and len(X_test) == len(y_test)
assert np.issubdtype(y_train.dtype, np.number)
assert np.all(y_train == y_train.astype(int))
assert set(np.unique(y_test)).issubset(set(np.unique(y_train)))
labels = np.sort(np.unique(y_train)).astype(int)
class_labels = [CLASS_NAMES.get(int(k), "Class " + str(k)) for k in labels]
assert len(labels) >= 2, "At least two classes are needed."
print("Training:", X_train.shape, X_train.dtype, "| Test:", X_test.shape, X_test.dtype)
print("Training labels:", labels.tolist(), "| Test retained for final evaluation.")
print("Images are 28x28 RGB if shapes match the assignment description.")

In [ ]:
# Class statistics come only from the supplied training partition.
counts = pd.Series(y_train.astype(int)).value_counts().reindex(labels, fill_value=0)
summary = pd.DataFrame({
    "class_id": labels,
    "class_name": class_labels,
    "train_count": counts.values,
    "train_fraction": (counts.values / len(y_train)).round(4),
})
display(summary)
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.bar(summary["class_id"].astype(str), summary["train_count"])
ax.set(xlabel="Training class ID", ylabel="Number of patches",
       title="Canvas training-subset class distribution")
plt.tight_layout()
plt.show()
print("Majority-class training proportion:", round(float(counts.max() / len(y_train)), 4))

### Examples of preprocessed data
Pixels are converted to float32 and rescaled to [0,1] using a training-informed 255 or 1 divisor. Both the trivial and geometric baselines use the same flattened RGB vectors. There are no learned preprocessing statistics; image augmentation and stain normalisation are deliberately omitted for a reproducible reference.

In [ ]:
# Image-level re-scaling uses a fixed divisor; no test-set statistics are fitted.
train_min, train_max = float(np.min(X_train)), float(np.max(X_train))
assert np.isfinite(train_min) and np.isfinite(train_max), "Non-finite training pixels."
assert train_min >= 0, "Expected nonnegative RGB intensities."
assert train_max <= 255.0 + 1e-6, "Unexpected pixel range; review scaling."
PIXEL_DIVISOR = 255.0 if train_max > 1.0 + 1e-6 else 1.0
print(f"Train pixel range: [{train_min:.4f}, {train_max:.4f}]; divisor: {PIXEL_DIVISOR}")
all_indices = np.arange(len(y_train))
fit_idx, val_idx = train_test_split(
    all_indices, test_size=0.20, random_state=SEED,
    stratify=y_train.astype(int)
)
assert len(set(fit_idx).intersection(set(val_idx))) == 0
y_fit, y_val = y_train[fit_idx].astype(int), y_train[val_idx].astype(int)
X_fit = np.asarray(X_train[fit_idx], dtype=np.float32).reshape(len(fit_idx), -1) / PIXEL_DIVISOR
X_val = np.asarray(X_train[val_idx], dtype=np.float32).reshape(len(val_idx), -1) / PIXEL_DIVISOR
print(f"Development split: fit={len(fit_idx)}, validation={len(val_idx)}, "
      f"held-out test={len(y_test)}")
print("Flattened features:", X_fit.shape[1])
assert X_fit.min() >= 0 and X_fit.max() <= 1.0 + 1e-6
assert X_val.min() >= 0 and X_val.max() <= 1.0 + 1e-6

In [ ]:
# Show genuine samples from the fit partition, not test examples.
fig, axes = plt.subplots(len(labels), 3, figsize=(7.5, 2.4 * len(labels)))
for row, label in enumerate(labels):
    candidates = fit_idx[y_fit == label]
    chosen = np.random.default_rng(SEED + int(label)).choice(
        candidates, size=min(3, len(candidates)), replace=False
    )
    for column in range(3):
        ax = axes[row, column]
        ax.axis("off")
        if column < len(chosen):
            rgb = np.asarray(X_train[chosen[column]], dtype=np.float32) / PIXEL_DIVISOR
            ax.imshow(np.clip(rgb, 0, 1))
        if column == 0:
            ax.set_title(CLASS_NAMES.get(int(label), "Class " + str(label)),
                         fontsize=9, loc="left")
plt.suptitle("Training examples (raw RGB intensities rescaled for display)")
plt.tight_layout()
plt.show()

## 2. Algorithm design and setup
**Assessment boundary:** the following are diagnostic reference algorithms. The assigned first-six-weeks method and the two neural networks are implemented in their own notebooks.

### Algorithm of choice from first six weeks of course
To be completed in `Specific_Algorithm_Rememeber_to_change_name.ipynb` by its owner; not replaced by this reference baseline.

### Fully connected neural network
To be completed in `MLP.ipynb` by its owner.

### Convolutional neural network
To be completed in `CNN.ipynb` by its owner.

### Reference baselines and evaluation policy
**Dummy most-frequent** measures the class-imbalance floor; **nearest centroid** compares flattened 28×28 RGB images by Euclidean distance from each class training mean. The latter discards spatial invariance and may underfit highly variable textures. Both are intentionally cheap, transparent reference points; neither is the team's assessed first-six-weeks classifier.

**Primary metric:** macro-F1, which gives each class equal influence. **Secondary:** accuracy, balanced accuracy, per-class precision/recall/F1, confusion matrix. Class order is fixed from training labels. Record fit time and prediction time separately on the same hardware, and avoid strict cross-machine speed claims. The validation split is for model comparisons and hyperparameters; test is used only for final fixed-config evaluation.

In [ ]:
# Shared measurement protocol. Macro-F1 is the primary selection metric.
PRIMARY_METRIC = "macro_f1"
def measure(y_true, y_pred):
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, labels=labels,
                                   average="macro", zero_division=0)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred))
    }

def make_baseline(name):
    if name == "Most frequent":
        return DummyClassifier(strategy="most_frequent")
    if name == "Nearest centroid (RGB pixels)":
        return NearestCentroid(metric="euclidean")
    raise ValueError("Unknown baseline: " + name)

BASELINE_NAMES = ["Most frequent", "Nearest centroid (RGB pixels)"]
print("Model-independent protocol:",
      "stratified validation from original training data, seed", SEED,
      "; primary: macro-F1; secondary: accuracy and balanced accuracy.")
print("Train and prediction times use perf_counter; include fit/predict calls, "
      "exclude dataset loading and plotting.")

## 3. Hyperparameter tuning
For the two extra reference baselines only, configurations are fixed **without hyperparameter search**. The three required models must separately perform the assignment's tuning; MLP and CNN each require at least three distinct hyperparameters. Do not substitute this descriptive comparison for that requirement.

### Algorithm of choice from first six weeks of course
See the owner-specific notebook. Tuning output must include parameter combinations and wall-clock times.

### Fully connected neural network
See `MLP.ipynb`; independently search at least three hyperparameters.

### Convolutional neural network
See `CNN.ipynb`; independently search at least three hyperparameters.

### Reference validation comparison (not a hyperparameter search)

In [ ]:
# This cell measures fixed reference algorithms; it is NOT a search for the
# required three assessed algorithms, and makes no decisions using the test set.
validation_records = []
for model_name in BASELINE_NAMES:
    model = make_baseline(model_name)
    fit_start = time.perf_counter()
    model.fit(X_fit, y_fit)
    fit_seconds = time.perf_counter() - fit_start
    pred_start = time.perf_counter()
    y_val_pred = model.predict(X_val)
    prediction_seconds = time.perf_counter() - pred_start
    validation_records.append({
        "model": model_name, **measure(y_val, y_val_pred),
        "fit_seconds": fit_seconds, "prediction_seconds": prediction_seconds
    })
validation_results = pd.DataFrame(validation_records).sort_values(
    "macro_f1", ascending=False
).reset_index(drop=True)
display(validation_results.round(4))
print("No hyperparameters are optimised here; validation is descriptive only.")
print("Any future choice of assessed-model hyperparameters must use this validation split, "
      "not test-set performance.")

In [ ]:
ax = validation_results.set_index("model")[["accuracy", "macro_f1", "balanced_accuracy"]].plot(
    kind="bar", figsize=(9, 4), ylim=(0, 1), rot=0
)
ax.set(title="Reference baselines on validation (same split)",
       ylabel="Metric value", xlabel="")
plt.tight_layout()
plt.show()

## 4. Final models
Final reference models must be trained from scratch in a standalone cell. They use **all original Canvas training examples** with the predeclared configurations, and test once on the held-out Canvas test set. Skipping the optional validation comparison cells does not break these final cells.

### Algorithm of choice from first six weeks of course
Integrate the actual owner model here in the final submitted notebook.

### Fully connected neural network
Integrate the independently re-trained final MLP here.

### Convolutional neural network
Integrate the independently re-trained final CNN here.

### Reference-only final evaluation

In [ ]:
# This section is independent of the validation cell: no search object or
# validation-prediction variable is required. Reference configurations are frozen.
# Fit on ALL original training examples after defining the protocol above.
X_full = np.asarray(X_train, dtype=np.float32).reshape(len(X_train), -1) / PIXEL_DIVISOR
X_test_features = np.asarray(X_test, dtype=np.float32).reshape(len(X_test), -1) / PIXEL_DIVISOR
assert X_full.shape[1] == X_test_features.shape[1]
test_records = []
final_predictions = {}
for model_name in BASELINE_NAMES:
    model = make_baseline(model_name)
    start_fit = time.perf_counter()
    model.fit(X_full, y_train.astype(int))
    elapsed_fit = time.perf_counter() - start_fit
    start_predict = time.perf_counter()
    prediction = model.predict(X_test_features)
    elapsed_predict = time.perf_counter() - start_predict
    final_predictions[model_name] = prediction
    test_records.append({
        "model": model_name, "frozen_parameters": str(model.get_params()),
        **measure(y_test.astype(int), prediction),
        "fit_seconds": elapsed_fit,
        "prediction_seconds": elapsed_predict,
        "train_samples": len(y_train), "test_samples": len(y_test),
        "seed": SEED
    })
baseline_test_results = pd.DataFrame(test_records).sort_values(
    "macro_f1", ascending=False
).reset_index(drop=True)
display(baseline_test_results.round(4))
print("The reported test metrics are single-split reference measurements, "
      "not validation tuning scores.")

In [ ]:
# Diagnostic predictions for the fixed nearest-centroid reference.
chosen_name = "Nearest centroid (RGB pixels)"
chosen_predictions = final_predictions[chosen_name]
print(classification_report(
    y_test.astype(int), chosen_predictions, labels=labels,
    target_names=class_labels, zero_division=0, digits=3
))
cm = confusion_matrix(y_test.astype(int), chosen_predictions, labels=labels)
fig, ax = plt.subplots(figsize=(9, 7))
ConfusionMatrixDisplay(cm, display_labels=[str(i) for i in labels]).plot(
    ax=ax, cmap="Blues", colorbar=False, values_format="d"
)
ax.set_title("Fixed reference: nearest-centroid test confusion matrix")
ax.set_xlabel("Predicted class ID")
ax.set_ylabel("Actual class ID")
plt.tight_layout()
plt.show()
print("Interpretation caution: neither temporal nor patient-level grouping "
      "is verifiable from the supplied arrays alone.")
print("Cross-machine elapsed times are not controlled hardware benchmarks.")

### Interpretation and limitations
This is a small Canvas subset rather than the complete publicly released PathMNIST benchmark. Reference performance must not be compared directly with published full-dataset scores. A stratified image-level split cannot guarantee patient- or slide-level independence, because no group IDs are supplied. The low-resolution patch setting can erase texture detail. Nearest-centroid distances in raw pixels are sensitive to stain and colour variation, while a majority predictor disregards images entirely. Macro-F1/accuracy and runtimes are single-run summaries, not confidence intervals. Device-dependent runtime comparisons require matching hardware and software.

## 5. AI Acknowledgement
ChatGPT (OpenAI) assisted in preparing the notebook structure, reference baseline implementation, protocol explanation, and initial documentation. The team must review the outputs, verify the analysis and update the final acknowledgement to reflect actual additional usage.

**Dataset/class labels:** course Canvas subset; PathMNIST labels from the official MedMNIST `info.py` repository (https://github.com/MedMNIST/MedMNIST/blob/main/medmnist/info.py).